# 01 — Prepare and validate citation states

**Purpose:** inspect the citation records used as model inputs before spending time or money on inference. A state is a JSON object containing `study_id`, `title`, and `abstract`.

| Item | Description |
|---|---|
| Inputs | Dataset selected by the evaluation YAML; default `data/examples/states.jsonl` |
| Outputs | Validated states, counts, warnings, reproducible sample and checksum; optional private CSV |
| Parameters | Dataset `path`, `root_env`, `format`, `glob`, `expected_count`; seed and notebook record limit |

JSON stores one structured object; JSONL stores one object per line. The loader also accepts a JSON array. The default real-data profile expects a directory of individual JSON files. No API calls are made in this notebook.

## 1. Load project settings

Use the synthetic profile first. To select private data later, change `evaluation_config` in `configs/notebooks.yaml`. Keep real records under `data/private/` or outside the repository; an optional `JEV_DATA_DIR` environment variable overrides the real-data path.

In [ ]:
try:
    from jev_eval.environment import initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select this project's .venv Python kernel. From the repository terminal, "
        "run: uv sync --locked --group notebooks"
    ) from None

project = initialize_notebook()

## 2. Load and inspect counts

The loader checks that files exist and that their JSON can be read. An exact `expected_count`, when configured, is checked against the complete input before sampling. Errors identify a file and line without echoing citation text.

In [ ]:
from jev_eval.data import load_states, summarize_states, validate_states

rows = load_states(project.evaluation.dataset, project.root)
validation = validate_states(rows)
summarize_states(validation)

## 3. Review validation issues

Duplicate IDs, non-text fields, unexpected fields, or records with neither a title nor an abstract are errors. Missing abstracts are warnings: they do not automatically remove citations or determine systematic-review eligibility. Review these records according to the screening protocol.

The issue report uses row numbers and field names, not citation text. Resolve errors in the source data before proceeding. Do not edit the original dataset in a notebook cell.

In [ ]:
import pandas as pd

pd.DataFrame(validation.issues, columns=["row", "field", "severity", "issue"])

In [ ]:
states = validation.require_valid()
print(f"Validated {len(states)} states; no records silently excluded.")

## 4. Create a reproducible small sample

The seed and sample limit control a local test subset. The complete dataset remains unchanged. A checksum records the normalized records and their order without displaying their contents.

In [ ]:
from jev_eval.data import dataset_fingerprint, select_sample

sample = select_sample(states, project.workflow.execution.max_records, project.evaluation.seed)
print(f"Sample: {len(sample)} states")
print("Complete dataset SHA-256:", dataset_fingerprint(states))

## 5. Optional text inspection — may display private data

The next cell displays citation text only if `privacy.display_record_text` is true in notebook settings. Leave it false for routine shared workflows. If you choose to inspect text, it may be saved in notebook outputs by Jupyter; clear all outputs before committing.

In [ ]:
if project.workflow.privacy.display_record_text:
    display(summarize_states(validation, include_text=True))
else:
    print("Citation text display is disabled; aggregate checks are available above.")

## 6. Optional export — writes a file

**File-writing warning:** enabling the next cell creates a CSV under ignored `results/`. It does not modify the dataset. Enable `output.allow_writes` in YAML and set `ALLOW_WRITES` below to true. Existing files are never overwritten by this export function.

In [ ]:
from jev_eval.reporting import export_table

ALLOW_WRITES = False
if ALLOW_WRITES:
    export_table(
        summarize_states(validation),
        project,
        "results/data-validation/summary.csv",
        allow_writes=True,
    )
else:
    print("Export skipped; no files written.")

## 7. Next step

Proceed to `02_model_connection.ipynb`. That notebook reloads settings and inputs, so keeping this kernel open is unnecessary.